# 1. Funções de custo

Vamos começar comparando as funções de custo para verificar qual consegue ajustar melhor a curva. Para esses testes vamos começar usando uma normalização padrão, o min-max, e um valor padrão de x0 e alpha. Seja $e_i = y_i - \hat{y}_i$ o erro (resíduo) da amostra $i$ e $n$ o número total de amostras. As funções de custo que serão testadas serão:



| Função | J(e) | J'(e) | Jw |
| --- | --- | --- | --- |
| MSE | $\dfrac{1}{n}\displaystyle\sum_{i=1}^{n} e_i^{2}$ | $\dfrac{2}{n} e_i$ | $- \dfrac{2}{n} \bar{F} ^ {T} e_i$ |
| RMSE | $\sqrt{MSE}$ | $\dfrac{1}{ 2 \sqrt{MSE}}$ | $\dfrac{J_w MSE}{ 2 \sqrt{MSE}}$ |
| MAE | $\dfrac{1}{n}\displaystyle\sum_{i=1}^{n} \lvert e_i \rvert$ | $\dfrac{1}{n}\,\mathrm{sinal}(e_i)$ | $-\dfrac{1}{n} \bar{F} ^ {T} \mathrm{sinal}(e_i)$|

In [22]:
import numpy as np
import pandas as pd


dados = pd.read_excel("Trabalho2dados.xlsx")
x_raw = dados["x"].to_numpy()
y_raw = dados["y"].to_numpy()
z_raw = dados["z"].to_numpy()

def min_max_normaliza(v):
    v_min, v_max = v.min(), v.max()
    return 2 * (v - v_min) / (v_max - v_min) - 1, v_min, v_max

def min_max_desnormaliza(v_norm, v_min, v_max):
    return (v_norm + 1) / 2 * (v_max - v_min) + v_min

x, x_min, x_max = min_max_normaliza(x_raw)
y, y_min, y_max = min_max_normaliza(y_raw)
z, z_min, z_max = min_max_normaliza(z_raw)

n = len(z)
F_bar = np.column_stack((x**3, y**2, np.ones(n)))

def modelo(theta, F_bar):
    return F_bar @ theta

def erro(theta, F_bar, z):
    return z - modelo(theta, F_bar)

# --- FUNÇÕES DE CUSTO E GRADIENTES CORRIGIDOS ---

# MSE
def J_MSE(theta, F_bar, z):
    e = erro(theta, F_bar, z)
    return np.mean(e**2)

def gradiente_J_MSE(theta, F_bar, z):
    e = erro(theta, F_bar, z)
    n = F_bar.shape[0]
    return - (2 / n) * (F_bar.T @ e)

# RMSE
def J_RMSE(theta, F_bar, z):
    return np.sqrt(J_MSE(theta, F_bar, z))

def gradiente_J_RMSE(theta, F_bar, z):
    rmse = J_RMSE(theta, F_bar, z)
    if rmse == 0:
        return np.zeros_like(theta)
    return gradiente_J_MSE(theta, F_bar, z) / (2 * rmse)

# MAE (Corrigido: np.mean já contempla a divisão por n)
def J_MAE(theta, F_bar, z):
    e = erro(theta, F_bar, z)
    return np.mean(np.abs(e))

def gradiente_J_MAE(theta, F_bar, z):
    e = erro(theta, F_bar, z)
    return - (1 / n) * (F_bar.T @ np.sign(e))

# --- DESCIDA DO GRADIENTE ---

def descida_gradiente(J, gradiente_J, alpha, x0, args=(), max_iter=5000, tol=1e-8):
    x = np.array(x0, dtype=float)
    x_hist, hist = [], []

    Jx = J(x, *args)
    hist.append(f"Iteração 0 : x = {x} -> J(x) = {Jx:.6f}\n")
    x_hist.append(x.copy())

    for i in range(max_iter):
        grad = gradiente_J(x, *args)
        xi = x - (alpha * grad)

        Jxi = J(xi, *args)
        Jx = J(x, *args)

        if np.linalg.norm(xi - x) <= tol or abs(Jxi - Jx) <= tol:
            hist.append(f"Iteração {i + 1} : x = {xi} -> J(x) = {Jxi:.6f} [Convergiu]\n")
            x_hist.append(xi.copy())
            return xi, hist, x_hist

        x = xi
        hist.append(f"Iteração {i + 1} : x = {x} -> J(x) = {Jxi:.6f}\n")
        x_hist.append(xi.copy())

    return x, hist, x_hist

# Otimização
x0 = np.array([0.0, 0.0, 0.0])
alpha = 0.05

funcoes_custo = {
    "MSE": (J_MSE, gradiente_J_MSE),
    "RMSE": (J_RMSE, gradiente_J_RMSE),
    "MAE": (J_MAE, gradiente_J_MAE),
}

resultados = {}
for nome, (J, gradiente_J) in funcoes_custo.items():
    theta_opt, hist, x_hist = descida_gradiente(
        J, gradiente_J, alpha, x0, args=(F_bar, z), max_iter=max_iter, tol=tol
    )
    resultados[nome] = {"theta": theta_opt, "hist": hist, "x_hist": np.array(x_hist)}
    convergiu = "[Convergiu]" in hist[-1]
    print(
        f"{nome:<5} -> {len(x_hist) - 1:5d} iterações "
        f"(convergiu={convergiu}) -> theta* = {theta_opt} -> J(theta*) = {J(theta_opt, F_bar, z):.6f}"
    )

MSE   ->   404 iterações (convergiu=True) -> theta* = [0.87525653 0.20521131 0.00631333] -> J(theta*) = 0.003020
RMSE  ->    90 iterações (convergiu=True) -> theta* = [0.87527327 0.20682537 0.00555613] -> J(theta*) = 0.054950
MAE   -> 20000 iterações (convergiu=False) -> theta* = [0.86238426 0.19598765 0.01851852] -> J(theta*) = 0.044898


In [23]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


grid_x = np.linspace(x_raw.min(), x_raw.max(), 40)
grid_y = np.linspace(y_raw.min(), y_raw.max(), 40)
Gx, Gy = np.meshgrid(grid_x, grid_y)

Gx_norm = 2 * (Gx - x_min) / (x_max - x_min) - 1
Gy_norm = 2 * (Gy - y_min) / (y_max - y_min) - 1
F_bar_grid = np.column_stack((Gx_norm.ravel()**3, Gy_norm.ravel()**2, np.ones(Gx_norm.size)))

fig = make_subplots(
    rows=2, cols=2,
    specs=[[{'type': 'scene'}, {'type': 'scene'}],
           [{'type': 'scene'}, {'type': 'scene'}]],
    subplot_titles=("MSE", "RMSE", "MAE")
)

posicoes = {"MSE": (1, 1), "RMSE": (1, 2), "MAE": (2, 1)}
cores = {"MSE": "#1f77b4", "RMSE": "#ff7f0e", "MAE": "#2ca02c"}

for nome, (row, col) in posicoes.items():
    # Corrigido: acessa especificamente a chave "theta"
    theta_opt = resultados[nome]["theta"]
    
    Gz_norm = modelo(theta_opt, F_bar_grid).reshape(Gx_norm.shape)
    Gz = min_max_desnormaliza(Gz_norm, z_min, z_max)
    
    fig.add_trace(
        go.Scatter3d(
            x=x_raw, y=y_raw, z=z_raw, mode="markers",
            marker=dict(size=2, color="black"), showlegend=False
        ),
        row=row, col=col
    )
    
    fig.add_trace(
        go.Surface(
            x=Gx, y=Gy, z=Gz, opacity=0.6, showscale=False,
            colorscale=[[0, cores[nome]], [1, cores[nome]]],
            showlegend=False
        ),
        row=row, col=col
    )

fig.update_layout(
    title="Comparação de Superfícies Ajustadas por Função de Custo",
    height=800,
    width=1000
)

fig.show()

In [24]:
import numpy as np
import pandas as pd

def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot)

# Avaliação das Funções de Custo na escala original
print(f"{'Função':<8} {'iterações':>10} {'R²':>12}")
print("-" * 32)

resultados_custo = {}

for nome in funcoes_custo.keys():
    theta_opt = resultados[nome]["theta"]
    hist = resultados[nome]["hist"]
    iters = len(resultados[nome]["x_hist"]) - 1
    convergiu = "[Convergiu]" in hist[-1]
    
    # 1. Previsão no espaço normalizado
    z_pred_norm = modelo(theta_opt, F_bar)
    
    # 2. Desnormalização para a escala real (z_raw)
    z_pred_orig = min_max_desnormaliza(z_pred_norm, z_min, z_max)
    
    # 3. Métricas na escala original
    r2_orig = r2_score(z_raw, z_pred_orig)
    
    resultados_custo[nome] = {
        "r2": r2_orig,
        "iters": iters,
    }
    
    print(f"{nome:<8} {iters:>10d} {r2_orig:>12.5f}")

Função    iterações           R²
--------------------------------
MSE             404      0.98575
RMSE             90      0.98576
MAE           20000      0.98514


# 2. Normalização

Com a função de custo já definida (RMSE), agora precisamos definir a normalização. Foram testados três métodos de normalização para $x$, $y$ e $z$, mantendo `x0` e `alpha` fixos:

| Método | Fórmula |
| --- | --- |
| Min-max | $v' = \dfrac{v - v_{min}}{v_{max} - v_{min}}$ |
| Padrão (normalização de média) | $v' = \dfrac{v - \bar{v}}{v_{max} - v_{min}}$ |
| Z-score | $v' = \dfrac{v - \bar{v}}{\sigma_v}$ |

In [25]:
def padrao_normaliza(v):
    v_mean, v_min, v_max = v.mean(), v.min(), v.max()
    return (v - v_mean) / (v_max - v_min), v_mean, v_min, v_max

def padrao_desnormaliza(v_norm, v_mean, v_min, v_max):
    return v_norm * (v_max - v_min) + v_mean

def zscore_normaliza(v):
    v_mean, v_std = v.mean(), v.std()
    return (v - v_mean) / v_std, v_mean, v_std

def zscore_desnormaliza(v_norm, v_mean, v_std):
    return v_norm * v_std + v_mean

def construir_F_bar(xn, yn):
    return np.column_stack((xn**3, yn**2, np.ones(len(xn))))

x_padrao, x_padrao_mean, x_padrao_min, x_padrao_max = padrao_normaliza(x_raw)
y_padrao, y_padrao_mean, y_padrao_min, y_padrao_max = padrao_normaliza(y_raw)
z_padrao, z_padrao_mean, z_padrao_min, z_padrao_max = padrao_normaliza(z_raw)

x_zscore, x_zscore_mean, x_zscore_std = zscore_normaliza(x_raw)
y_zscore, y_zscore_mean, y_zscore_std = zscore_normaliza(y_raw)
z_zscore, z_zscore_mean, z_zscore_std = zscore_normaliza(z_raw)

esquemas_normalizacao = {
    "min-max": {
        "F_bar": F_bar, "z": z,
        "desnormaliza_z": lambda v: min_max_desnormaliza(v, z_min, z_max),
    },
    "padrão": {
        "F_bar": construir_F_bar(x_padrao, y_padrao), "z": z_padrao,
        "desnormaliza_z": lambda v: padrao_desnormaliza(v, z_padrao_mean, z_padrao_min, z_padrao_max),
    },
    "z-score": {
        "F_bar": construir_F_bar(x_zscore, y_zscore), "z": z_zscore,
        "desnormaliza_z": lambda v: zscore_desnormaliza(v, z_zscore_mean, z_zscore_std),
    },
}


In [26]:
x0 = np.array([0.0, 0.0, 0.0])
alpha = 0.05

resultados_normalizacao = {}
print(f"{'Método':<8} {'iterações':>10}  {'R²':>10}")
for nome, esquema in esquemas_normalizacao.items():
    F_bar_n, zn = esquema["F_bar"], esquema["z"]
    theta_opt, hist, x_hist = descida_gradiente(
        J_RMSE, gradiente_J_RMSE, alpha, x0, args=(F_bar_n, zn))
    z_pred = esquema["desnormaliza_z"](modelo(theta_opt, F_bar_n))
    r2 = r2_score(z_raw, z_pred)
    rmse_original = np.sqrt(np.mean((z_raw - z_pred) ** 2))
    resultados_normalizacao[nome] = {
        "theta": theta_opt, "iters": len(x_hist) - 1, "r2": r2,
    }
    print(f"{nome:<8} {len(x_hist) - 1:>10d} {r2:>10.5f}")

melhor_normalizacao = max(resultados_normalizacao, key=lambda n: resultados_normalizacao[n]["r2"])
print(f"\nMelhor normalização: {melhor_normalizacao}")


Método    iterações          R²
min-max          90    0.98576
padrão         1701    0.98576
z-score          60    0.98576

Melhor normalização: z-score


# 3. X0

Agora precisamos avaliar o X0.

In [27]:
F_bar_zscore = construir_F_bar(x_zscore, y_zscore)

x0_candidatos = {
    "[0.0, 0.0, 0.0]": [0.0, 0.0, 0.0],
    "[1.0, 1.0, 1.0]": [1.0, 1.0, 1.0],
    "[-3.0, 3.0, -3.0]": [-3.0, 3.0, -3.0],
}
alpha = 0.05

resultados_x0 = {}
print(f"{'x0':<12} {'iterações':>10} {'theta*':>32} {'J(theta*)':>12}")
for nome, x0_i in x0_candidatos.items():
    theta_opt, hist, x_hist = descida_gradiente(
        J_RMSE, gradiente_J_RMSE, alpha, x0_i,
        args=(F_bar_zscore, z_zscore))
    Jval = J_RMSE(theta_opt, F_bar_zscore, z_zscore)
    resultados_x0[nome] = {"theta": theta_opt, "iters": len(x_hist) - 1, "J": Jval}
    print(f"{nome:<12} {len(x_hist) - 1:>10d} {str(np.round(theta_opt, 4)):>32} {Jval:>12.6f}")

melhor_x0 = min(resultados_x0, key=lambda n: resultados_x0[n]["iters"])
print(f"\nMelhor x0 (menos iterações): {melhor_x0}")


x0            iterações                           theta*    J(theta*)
[0.0, 0.0, 0.0]         60        [ 0.5113  0.1872 -0.1872]     0.119344
[1.0, 1.0, 1.0]         79        [ 0.5113  0.1872 -0.1872]     0.119344
[-3.0, 3.0, -3.0]        208        [ 0.5113  0.1874 -0.1874]     0.119344

Melhor x0 (menos iterações): [0.0, 0.0, 0.0]


# 4. Alpha

Agora só precisamos testar até encontrar o melhor valor de alpha para a nossa função.

In [28]:
x0 = np.array([0.0, 0.0, 0.0])
alphas_teste = [0.025, 0.05, 0.075]

resultados_alpha = {}
print(f"{'alpha':<8} {'iterações':>10} {'convergiu':>10} {'theta*':>32} {'J(theta*)':>12}")
for alpha_i in alphas_teste:
    theta_opt, hist, x_hist = descida_gradiente(
        J_RMSE, gradiente_J_RMSE, alpha_i, x0,
        args=(F_bar_zscore, z_zscore))
    convergiu = "[Convergiu]" in hist[-1]
    Jval = J_RMSE(theta_opt, F_bar_zscore, z_zscore)
    resultados_alpha[alpha_i] = {"theta": theta_opt, "iters": len(x_hist) - 1, "J": Jval}
    print(f"{alpha_i:<8} {len(x_hist) - 1:>10d} {str(convergiu):>10} {str(np.round(theta_opt, 4)):>32} {Jval:>12.6f}")

melhor_alpha = min(resultados_alpha, key=lambda a: resultados_alpha[a]["J"])
print(f"\nMelhor alpha: {melhor_alpha}")


alpha     iterações  convergiu                           theta*    J(theta*)
0.025           116       True        [ 0.5113  0.1872 -0.1871]     0.119344
0.05             60       True        [ 0.5113  0.1872 -0.1872]     0.119344
0.075            45       True        [ 0.4756  0.1873 -0.1873]     0.137531

Melhor alpha: 0.05


In [29]:
theta, hist, x_hist = descida_gradiente(J_RMSE, gradiente_J_RMSE, 0.05, [0.0, 0.0, 0.0],args=(F_bar_zscore, z_zscore))

z_pred_final = zscore_desnormaliza(modelo(theta, F_bar_zscore), z_zscore_mean, z_zscore_std)
r2_final = r2_score(z_raw, z_pred_final)
rmse_final = np.sqrt(np.mean((z_raw - z_pred_final) ** 2))

print(f"theta* = {theta}")
print(f"iterações = {len(x_hist) - 1}")
print(f"R² = {r2_final:.5f}")
print(f"RMSE (escala original) = {rmse_final:.3f}")

grid_x = np.linspace(x_raw.min(), x_raw.max(), 40)
grid_y = np.linspace(y_raw.min(), y_raw.max(), 40)
Gx, Gy = np.meshgrid(grid_x, grid_y)
Gx_zscore = (Gx - x_zscore_mean) / x_zscore_std
Gy_zscore = (Gy - y_zscore_mean) / y_zscore_std
F_bar_grid_zscore = construir_F_bar(Gx_zscore.ravel(), Gy_zscore.ravel())
Gz = zscore_desnormaliza(modelo(theta, F_bar_grid_zscore), z_zscore_mean, z_zscore_std).reshape(Gx.shape)

fig = go.Figure()
fig.add_trace(go.Scatter3d(x=x_raw, y=y_raw, z=z_raw, mode="markers", marker=dict(size=3, color="black"), name="dados"))
fig.add_trace(go.Surface(x=Gx, y=Gy, z=Gz, opacity=0.6, showscale=False, colorscale="Viridis", name="ajuste final"))
fig.update_scenes(xaxis_title="x", yaxis_title="y", zaxis_title="z")
fig.update_layout(title="Ajuste final: RMSE + z-score", height=650, width=900, legend=dict(orientation="h", y=-0.05))
fig.show()

theta* = [ 0.51128731  0.18724329 -0.1872081 ]
iterações = 60
R² = 0.98576
RMSE (escala original) = 34.951
